In [1]:
import os
import sys
import torch
from tqdm import tqdm
from dotenv import load_dotenv
import pandas as pd
load_dotenv()
sys.path.append(os.getenv("ROOT_PATH"))
import miso_utils.datasets as mud

In [8]:
val_data = mud.MisoValDataset(os.getenv("VAL_PATH"))
train_data = mud.MisoTrainDataset(os.getenv("TRAIN_PATH"))

Files Left: 600it [00:03, 164.99it/s]
Files Left: 9395it [02:47, 56.10it/s] 


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MisoCNN(nn.Module):
    def __init__(self):
        super(MisoCNN, self).__init__()
        # Input: 3 x 224 x 224
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        
        # After 3 pools, 224 -> 112 -> 56 -> 28
        self.fc1 = nn.Linear(128 * 28 * 28, 512)
        self.fc2 = nn.Linear(512, 1) # Single output for binary (0 or 1)
        
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        
        x = x.view(-1, 128 * 28 * 28) # Flatten
        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x)) # Sigmoid for binary probability
        return x

In [ ]:
from torch.utils.data import DataLoader
import torch.optim as optim
import os

# 1. Setup Data
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

# 2. Setup Model, Loss, and Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MisoCNN().to(device)
criterion = nn.BCELoss() # Binary Cross Entropy
optimizer = optim.Adam(model.parameters(), lr=0.003)

# 3. Training Loop
epochs = 20
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for batch in tqdm(train_loader):
        # Move data to CUDA
        images = batch['img'].to(device)
        labels = batch['indian_label'].to(device).float().unsqueeze(1)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")

    # 4. Basic Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in val_loader:
            images = batch['img'].to(device)
            labels = batch['indian_label'].to(device).float().unsqueeze(1)
            outputs = model(images)
            predictions = (outputs > 0.5).float()
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    
    print(f"Validation Accuracy: {100 * correct / total:.2f}%")